# Deep Learning for Business Applications course

## TOPIC 1: Introduction to Deep Learning. Backpropagation

*Notebook for MBA (Master of Business Analytics) students. No deep math required — only intuition, pictures, and running the code.*

Based on [this article](https://www.geeksforgeeks.org/backpropagation-in-neural-network/) and [this repository](https://github.com/jgabriellima/backpropagation).

## 1. The big picture: what are we actually doing?

A **neural network** is just a function with freely adjustable "knobs" (called **weights**).

- **Inputs** = data we already know (facts, features).
- **Outputs** = the answer we want (a prediction).
- **Training** = automatically turning the knobs until the predictions match the known answers.

Everyday analogy: think of a mixing desk in a recording studio. The signal flows through the desk, and there are dozens of faders. An engineer moves the faders, listens to the result, compares it with the "ideal sound", and adjusts again. A neural network does exactly this — thousands of times per second — with the knobs turning themselves.

**Backpropagation** is simply the algorithm that tells each knob *which way* to turn and *by how much*: measure how wrong the answer is, then distribute the "blame" backwards through the network, knob by knob.

## 2. The toy problem: XOR

XOR ("exclusive or") answers **1 when the two inputs are different**:

| Input A | Input B | XOR output |
|:-------:|:-------:|:----------:|
|    0    |    0    |      0     |
|    0    |    1    |      1     |
|    1    |    0    |      1     |
|    1    |    1    |      0     |

Why is this interesting for a manager? Because XOR is a simple picture of a **non-obvious rule**: the answer is not predictable from either input alone — only from their *combination*. Cases like "fraud happens only when amount is high AND country changes" have exactly this shape. If our tiny network can *learn* XOR from examples alone, the same mechanics can learn real business patterns from real data.

```text
inputs        hidden layer         output
                     w2
   x1 ──> ┌─────┐
    \──── │  h1 │─────\
     \    └─────┘      \
      w1                 w3 ──> y
     /    ┌─────┐      /
    /──── │  h2 │─────/
   x2 ──> └─────┘
                     w4
```

The middle "hidden layer" is what lets the network combine inputs — without it, XOR (and fraud detection) would be impossible.

## 3. Libraries

In [ ]:
import math
import random

import matplotlib.pyplot as plt
import numpy as np

## 4. Utility functions

Three small helpers:

- `rand(a, b)` — a random number between `a` and `b`; used to *initially* place the knobs.
- `matrix(i, j)` — an empty weight table.
- `afunc` — the **activation function**: a smooth "soft switch" (`tanh`) that squashes any number into the range −1..+1. Non-linearity like this is what allows the network to represent non-obvious rules. `dafunc` is its slope — needed later to correct the knobs.

In [ ]:
def rand(a, b):
    """
    Returns a random number between `a` and `b`

    """
    return (b - a) * random.random() + a


def matrix(i_size, j_size, fill=0):
    """
    Returns a matrix with size (i_size, j_size)

    """
    m = []
    for i in range(i_size):
        m.append([fill] * j_size)
    return m


def afunc(x):
    """
    Activation function — a smooth 'soft switch',
    here `tanh` instead of the classical sigmoid

    """
    return math.tanh(x)


def dafunc(y):
    """
    Derivative of the activation function
    in terms of the output (i.e. y)

    """
    return 1 - y ** 2

## 5. The neural network

Read the class below as a description of the mixing desk:

| Code | Business meaning |
|:-----|:-----------------|
| `self.wi`, `self.wo` | the knobs (weights) |
| `update(inputs)` | an *observation* flows through the desk → a prediction comes out |
| `back_propagate(...)` | compare prediction with the known answer, distribute the blame, turn every knob a little |
| `train(...)` | repeat thousands of times; `lr` = how big each turn is, `mm` = keep some of yesterday's direction (momentum) |

In [ ]:
class NN:
    def __init__(self, ni, nh, no):
        # number of input, hidden, and output nodes
        self.ni = ni + 1  # `+1` is for bias node
        self.nh = nh
        self.no = no

        # activations for nodes
        self.ai = [1] * self.ni
        self.ah = [1] * self.nh
        self.ao = [1] * self.no

        # create weights
        self.wi = matrix(self.ni, self.nh)
        self.wo = matrix(self.nh, self.no)
        # initialize them with random values
        for i in range(self.ni):
            for j in range(self.nh):
                self.wi[i][j] = rand(-0.2, 0.2)
        for j in range(self.nh):
            for k in range(self.no):
                self.wo[j][k] = rand(-2.0, 2.0)

        # last change in weights for momentum
        self.ci = matrix(self.ni, self.nh)
        self.co = matrix(self.nh, self.no)

    def update(self, inputs):
        """
        Returns outputs of the neural network

        """
        if len(inputs) != self.ni - 1:
            raise ValueError('wrong number of inputs')

        # input activations
        for i in range(self.ni - 1):
            self.ai[i] = inputs[i]

        # hidden activations
        for j in range(self.nh):
            hsum = 0
            for i in range(self.ni):
                hsum = hsum + self.ai[i] * self.wi[i][j]
            self.ah[j] = afunc(hsum)

        # output activations
        for k in range(self.no):
            osum = 0
            for j in range(self.nh):
                osum = osum + self.ah[j] * self.wo[j][k]
            self.ao[k] = afunc(osum)

        return self.ao[:]

    def back_propagate(self, targets, lr, mm):
        """
        Core of the backpropagation algorithm is here:
        error deltas are computed for the output layer,
        then propagated back to the hidden layer,
        and all weights are nudged accordingly

        """
        if len(targets) != self.no:
            raise ValueError('wrong number of target values')

        # calculate error terms for output
        output_deltas = [0] * self.no
        for k in range(self.no):
            error = targets[k] - self.ao[k]
            output_deltas[k] = dafunc(self.ao[k]) * error

        # calculate error terms for hidden
        hidden_deltas = [0] * self.nh
        for j in range(self.nh):
            error = 0
            for k in range(self.no):
                error = error + output_deltas[k] * self.wo[j][k]
            hidden_deltas[j] = dafunc(self.ah[j]) * error

        # update output weights
        for j in range(self.nh):
            for k in range(self.no):
                change = output_deltas[k] * self.ah[j]
                self.wo[j][k] = self.wo[j][k] + lr * change + mm * self.co[j][k]
                self.co[j][k] = change

        # update input weights
        for i in range(self.ni):
            for j in range(self.nh):
                change = hidden_deltas[j] * self.ai[i]
                self.wi[i][j] = self.wi[i][j] + lr * change + mm * self.ci[i][j]
                self.ci[i][j] = change

        # calculate error
        error = 0
        for k in range(len(targets)):
            error = error + .5 * (targets[k] - self.ao[k]) ** 2
        return error

    def test(self, patterns):
        """
        Prints outputs of the neural network
        for a test input pattern

        """
        for p in patterns:
            print(p[0], '->', self.update(p[0]))

    def weights(self):
        """
        Prints weights of the neural network

        """
        print('Input weights:')
        for i in range(self.ni):
            print(self.wi[i])
        print()
        print('Output weights:')
        for j in range(self.nh):
            print(self.wo[j])

    def train(self, patterns, iterations=1000, lr=.5, mm=0.1):
        """
        Train neural network with help
        of backpropagation algorithm

        :lr: learning rate
        :mm: momentum factor

        """
        self.error_history = []
        for i in range(iterations):
            error = 0
            for p in patterns:
                inputs = p[0]
                targets = p[1]
                self.update(inputs)
                error = error + self.back_propagate(targets, lr, mm)
            self.error_history.append(error)
            if i % 100 == 0:
                print('error at iteration {} is {:.4f}'.format(i, error))

## 6. Training on XOR

In [ ]:
# create a network with two input,
# two hidden, and one output nodes

nn = NN(2, 2, 1)

In [ ]:
# take XOR operations as a training data
pattern_xor = [
    [[0, 0], [0]],
    [[0, 1], [1]],
    [[1, 0], [1]],
    [[1, 1], [0]]
]

# ...and train it with some patterns
nn.train(pattern_xor)

### Loss curve: the learning "EKG"

The number above is the **loss** (error) — literally "how wrong is the network". When training goes well, it should fall. Watching the loss fall is how data science teams know their models are actually learning — the same job a dashboard does for a business KPI.

In [ ]:
plt.figure(figsize=(8, 3))
plt.plot(nn.error_history)
plt.title('Loss while training on XOR')
plt.xlabel('Training iteration')
plt.ylabel('Total error (loss)')
plt.grid(alpha=.3)
plt.show()

Notice the plateau at the start — the network is "looking for a grip" — and then a sharp drop. Real model training shows the same shape.

Now check what the network has learned (expect outputs close to 0, 1, 1, 0):

In [ ]:
# test our trained network
nn.test(pattern_xor)

## 7. Visualizing the learned knobs

Each line below is one weight (one "fader" position). Before training these are almost random; after training they encode the rule the network discovered. Look back at the diagram in section 2 if you want to trace which line is which knob.

In [ ]:
nn.weights()

## HOME ASSIGNMENT

Please, train neural network for **AND** and **OR** operations.

AND answers `1` only when both inputs are `1`; OR answers `1` when at least one input is `1` (tables are in the [bitwise-operators article](https://www.geeksforgeeks.org/complete-reference-for-bitwise-operators-in-programming-coding/)).

In [ ]:
# HINT:
# you need a new `pattern` for AND or OR
# operations only. No need to change the code
# for network, just train new NN with new pattern

pattern_new = [
    # Your code will be here
]
nn = NN(2, 2, 1)
nn.train(pattern_new)
nn.test(pattern_new)

In [ ]:
# Optional: draw the loss curve for your new pattern
plt.figure(figsize=(8, 3))
plt.plot(nn.error_history)
plt.title('Loss for my pattern')
plt.xlabel('Training iteration')
plt.ylabel('Total error (loss)')
plt.grid(alpha=.3)
plt.show()

## Additional questions to think about (no code needed)

1. Are AND and OR *easier* or *harder* than XOR to learn? Look at the loss curves and compare how quickly the error drops. Why might that be, in plain words?
2. What business problems "look like" XOR (a rule hidden in a combination of inputs), and which "look like" AND/OR (simple thresholds)?
3. If you start training from scratch (rerun the cells), do you always get the same result? Why is randomness tolerated in business ML pipelines, and when is it not?